#DAY 2 Databricks Challenge

##Challenges
### 🛠️ Tasks:

1. Upload sample e-commerce CSV
2. Read data into DataFrame
3. Perform basic operations: select, filter, groupBy, orderBy
4. Export results

###Creating a volume called challenge to store the relevant data

In [0]:
%sql
DROP VOLUME workspace.default.challenge

In [0]:
%sql
CREATE VOLUME workspace.default.challenge

###Reading CSV data (Upload .csv file into the path before running the command)
###Download **.csv** file here ---> [Link](https://github.com/ILAKKIYAN1994/DataScience/blob/daff9319cabc81f06ba451e98ac6dc1cb006f172/Databricks/14%20Days%20Challenge/Day%202/sample_commerce_events.csv)

In [0]:
df = spark.read.csv("/Volumes/workspace/default/challenge/sample_commerce_events.csv", header=True, inferSchema=True)
display(df)

event_time,event_type,product_id,product_name,category,brand,price,user_id
01-01-2025 10:01,view,101,iPhone 14,Electronics,Apple,999,1001
01-01-2025 10:02,cart,101,iPhone 14,Electronics,Apple,999,1001
01-01-2025 10:05,purchase,101,iPhone 14,Electronics,Apple,999,1001
01-01-2025 11:10,view,102,Galaxy S23,Electronics,Samsung,899,1002
01-01-2025 11:12,cart,102,Galaxy S23,Electronics,Samsung,899,1002
01-01-2025 11:15,purchase,102,Galaxy S23,Electronics,Samsung,899,1002
01-01-2025 12:00,view,103,MacBook Air,Electronics,Apple,1299,1003
01-01-2025 12:05,cart,103,MacBook Air,Electronics,Apple,1299,1003
01-01-2025 12:10,purchase,103,MacBook Air,Electronics,Apple,1299,1003
01-01-2025 13:00,view,104,Nike Shoes,Fashion,Nike,120,1004


###Optional to load the CSV data as a Parquet data into a specified location

In [0]:
df.write\
    .format("parquet")\
    .mode("overwrite")\
    .save("/Volumes/workspace/default/challenge/sample_commerce_events")

In [0]:
events = spark.read.parquet(
    "/Volumes/workspace/default/challenge/sample_commerce_events"
)
display(events)

event_time,event_type,product_id,product_name,category,brand,price,user_id
01-01-2025 10:01,view,101,iPhone 14,Electronics,Apple,999,1001
01-01-2025 10:02,cart,101,iPhone 14,Electronics,Apple,999,1001
01-01-2025 10:05,purchase,101,iPhone 14,Electronics,Apple,999,1001
01-01-2025 11:10,view,102,Galaxy S23,Electronics,Samsung,899,1002
01-01-2025 11:12,cart,102,Galaxy S23,Electronics,Samsung,899,1002
01-01-2025 11:15,purchase,102,Galaxy S23,Electronics,Samsung,899,1002
01-01-2025 12:00,view,103,MacBook Air,Electronics,Apple,1299,1003
01-01-2025 12:05,cart,103,MacBook Air,Electronics,Apple,1299,1003
01-01-2025 12:10,purchase,103,MacBook Air,Electronics,Apple,1299,1003
01-01-2025 13:00,view,104,Nike Shoes,Fashion,Nike,120,1004


In [0]:
events.printSchema()

root
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- user_id: integer (nullable = true)



###Select Columns

In [0]:
events.select("event_type","product_name","price").show(10)

+----------+------------+-----+
|event_type|product_name|price|
+----------+------------+-----+
|      view|   iPhone 14|  999|
|      cart|   iPhone 14|  999|
|  purchase|   iPhone 14|  999|
|      view|  Galaxy S23|  899|
|      cart|  Galaxy S23|  899|
|  purchase|  Galaxy S23|  899|
|      view| MacBook Air| 1299|
|      cart| MacBook Air| 1299|
|  purchase| MacBook Air| 1299|
|      view|  Nike Shoes|  120|
+----------+------------+-----+
only showing top 10 rows


###Filter records

In [0]:
events.filter("price > 120").count()

24

###Group By

In [0]:
events.groupBy("event_type").count().show()

+----------+-----+
|event_type|count|
+----------+-----+
|  purchase|   10|
|      cart|   10|
|      view|   10|
+----------+-----+



###Top 5 brands event wise

In [0]:
top_brands = events.groupBy("brand").count().orderBy("count", ascending=False).limit(5)
display(top_brands)

brand,count
Apple,9
Nike,6
Philips,3
Ikea,3
Adidas,3


###Exporting the Output

In [0]:
events.groupBy("event_type") \
      .count() \
      .write \
      .mode("overwrite") \
      .csv("/Volumes/workspace/default/challenge/output_sampecomm")


In [0]:
events.write \
  .mode("overwrite") \
  .option("header", "true") \
  .csv("/Volumes/workspace/default/challenge/output_sample_all")

###Verifying the output

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/challenge/output_sampecomm")


[FileInfo(path='dbfs:/Volumes/workspace/default/challenge/output_sampecomm/_SUCCESS', name='_SUCCESS', size=0, modificationTime=1767983875000),
 FileInfo(path='dbfs:/Volumes/workspace/default/challenge/output_sampecomm/_committed_5039490996990399720', name='_committed_5039490996990399720', size=113, modificationTime=1767983875000),
 FileInfo(path='dbfs:/Volumes/workspace/default/challenge/output_sampecomm/_started_5039490996990399720', name='_started_5039490996990399720', size=0, modificationTime=1767983875000),
 FileInfo(path='dbfs:/Volumes/workspace/default/challenge/output_sampecomm/part-00000-tid-5039490996990399720-7393bdf4-e7ce-4391-bb89-60bbfc2b2690-230-1-c000.csv', name='part-00000-tid-5039490996990399720-7393bdf4-e7ce-4391-bb89-60bbfc2b2690-230-1-c000.csv', size=28, modificationTime=1767983875000)]

In [0]:
events.groupBy("event_type").count()


DataFrame[event_type: string, count: bigint]

####Verifying grouped result

In [0]:
output1 = spark.read.csv("/Volumes/workspace/default/challenge/output_sampecomm", header=True, inferSchema=True)
display(output1)

purchase,10
cart,10
view,10


####Verifying whole result

In [0]:
output2 = spark.read.csv("/Volumes/workspace/default/challenge/output_sample_all/", header=True, inferSchema=True)
display(output2)

event_time,event_type,product_id,product_name,category,brand,price,user_id
01-01-2025 10:01,view,101,iPhone 14,Electronics,Apple,999,1001
01-01-2025 10:02,cart,101,iPhone 14,Electronics,Apple,999,1001
01-01-2025 10:05,purchase,101,iPhone 14,Electronics,Apple,999,1001
01-01-2025 11:10,view,102,Galaxy S23,Electronics,Samsung,899,1002
01-01-2025 11:12,cart,102,Galaxy S23,Electronics,Samsung,899,1002
01-01-2025 11:15,purchase,102,Galaxy S23,Electronics,Samsung,899,1002
01-01-2025 12:00,view,103,MacBook Air,Electronics,Apple,1299,1003
01-01-2025 12:05,cart,103,MacBook Air,Electronics,Apple,1299,1003
01-01-2025 12:10,purchase,103,MacBook Air,Electronics,Apple,1299,1003
01-01-2025 13:00,view,104,Nike Shoes,Fashion,Nike,120,1004


###Filtering Purchase events alone

In [0]:
events.filter("event_type = 'purchase'")\
    .select("product_name", "brand", "price")\
    .show(10)

+-------------+-------+-----+
| product_name|  brand|price|
+-------------+-------+-----+
|    iPhone 14|  Apple|  999|
|   Galaxy S23|Samsung|  899|
|  MacBook Air|  Apple| 1299|
|   Nike Shoes|   Nike|  120|
|Adidas Jacket| Adidas|  150|
|  AirPods Pro|  Apple|  249|
|  Smart Watch| Fitbit|  199|
| Office Chair|   Ikea|  299|
|   Table Lamp|Philips|   89|
|Running Shoes|   Nike|  140|
+-------------+-------+-----+



###Total Revenue by brand in descending order

In [0]:
from pyspark.sql.functions import sum

events.filter("event_type = 'purchase'") \
      .groupBy("brand") \
      .agg(sum("price").alias("total_revenue")) \
      .orderBy("total_revenue", ascending=False) \
      .show()


+-------+-------------+
|  brand|total_revenue|
+-------+-------------+
|  Apple|         2547|
|Samsung|          899|
|   Ikea|          299|
|   Nike|          260|
| Fitbit|          199|
| Adidas|          150|
|Philips|           89|
+-------+-------------+



###Most viewed products

In [0]:
events.filter("event_type = 'view'") \
      .groupBy("product_name") \
      .count() \
      .orderBy("count", ascending=False) \
      .limit(5) \
      .show()


+-------------+-----+
| product_name|count|
+-------------+-----+
|    iPhone 14|    1|
|Adidas Jacket|    1|
|  Smart Watch|    1|
|   Nike Shoes|    1|
|   Galaxy S23|    1|
+-------------+-----+



###Average Price by category

In [0]:
from pyspark.sql.functions import avg

events.groupBy("category") \
      .agg(avg("price").alias("avg_price")) \
      .orderBy("avg_price", ascending=False) \
      .show()


+-----------+------------------+
|   category|         avg_price|
+-----------+------------------+
|Electronics|             729.0|
|  Furniture|             194.0|
|    Fashion|136.66666666666666|
+-----------+------------------+



###High-Value Transactions

In [0]:
events.filter("event_type = 'purchase' AND price > 500") \
      .select("product_name", "brand", "price") \
      .show()


+------------+-------+-----+
|product_name|  brand|price|
+------------+-------+-----+
|   iPhone 14|  Apple|  999|
|  Galaxy S23|Samsung|  899|
| MacBook Air|  Apple| 1299|
+------------+-------+-----+



###Event Count per Category

In [0]:
events.groupBy("category") \
      .count() \
      .orderBy("count", ascending=False) \
      .show()


+-----------+-----+
|   category|count|
+-----------+-----+
|Electronics|   15|
|    Fashion|    9|
|  Furniture|    6|
+-----------+-----+



###Distinct Users per Brand

In [0]:
from pyspark.sql.functions import countDistinct #Need to import the countDistinct function to run
events.groupBy("brand") \
      .agg(countDistinct("user_id").alias("unique_users")) \
      .orderBy("unique_users", ascending=False) \
      .show()


+-------+------------+
|  brand|unique_users|
+-------+------------+
|  Apple|           3|
|   Nike|           2|
|Philips|           1|
| Adidas|           1|
|   Ikea|           1|
|Samsung|           1|
| Fitbit|           1|
+-------+------------+



###Add Discount Column


In [0]:
from pyspark.sql.functions import col #Its similar to calculated tables

events.withColumn("discounted_price", col("price") * 0.9) \
      .select("product_name", "price", "discounted_price") \
      .show(10)


+------------+-----+------------------+
|product_name|price|  discounted_price|
+------------+-----+------------------+
|   iPhone 14|  999|             899.1|
|   iPhone 14|  999|             899.1|
|   iPhone 14|  999|             899.1|
|  Galaxy S23|  899|             809.1|
|  Galaxy S23|  899|             809.1|
|  Galaxy S23|  899|             809.1|
| MacBook Air| 1299|1169.1000000000001|
| MacBook Air| 1299|1169.1000000000001|
| MacBook Air| 1299|1169.1000000000001|
|  Nike Shoes|  120|             108.0|
+------------+-----+------------------+
only showing top 10 rows


###Top Categories by Purchase Count

In [0]:
events.filter("event_type = 'purchase'") \
      .groupBy("category") \
      .count() \
      .orderBy("count", ascending=False) \
      .show()


+-----------+-----+
|   category|count|
+-----------+-----+
|Electronics|    5|
|    Fashion|    3|
|  Furniture|    2|
+-----------+-----+



## **For more such learning and insights**
- ### Follow me in [LinkedIn](https://www.linkedin.com/in/ilakkiyan-av/) 
- ### Follow me in [Youtube](https://www.youtube.com/@ilakkiyanav) 